**参考答案版**。请先独立完成 `lessons/` 里的练习,再来对照。

# Day 24 · 可复现性与版本治理：托管模型会悄悄改变

**心理大模型评测与 Agent** · 评测方法学 · 建议用时 3 到 4 小时

**今天的目标**

- 列出每次评测必须记录的信息，并写出 run manifest
- 用锚定集和控制图监测托管裁判的漂移
- 用哈希发现评测数据被改动
- 把方法学要求写成其他团队能照做的规范

**怎么用这个 notebook**：先看同一文件夹里的 `slides.html`，再从上到下运行这里的单元格。标着 **练习** 的单元格需要你自己填写，填完运行下面的批改单元格，会显示 `正确`、`再想想` 或 `待完成`。所有数据都是课程编写或模拟的，没有真实来访者数据。

In [ ]:
# 课程环境设置:让 notebook 找到课程代码库 src/course
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "course").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from course.plotting import use_style, OCHRE, TEAL, ROSE, SAND, INK
from course.check import check, check_true, summary
use_style()
pd.set_option("display.precision", 3)

## 1. 锚定集漂移监测

模拟 12 个月，每月让裁判给同一批 60 条回复打分。某个月起裁判被悄悄更新，整体打分偏宽松。

In [ ]:
from course.repro import anchor_drift, run_manifest, sha256_of
rng = np.random.default_rng(24)
CHANGE = int(rng.integers(5, 10))      # 变化发生的月份(从 0 开始),你不知道它
rows = []
for run in range(12):
    shift = 0.35 if run >= CHANGE else 0.0
    for i in range(60):
        rows.append({"run": run, "item": i, "score": 3.2 + shift + rng.normal(0, 0.7)})
hist = pd.DataFrame(rows)
mon = anchor_drift(hist, baseline_runs=4)
mon.round(3)

**练习 24.1**：第一个被标记的月份（`run`，从 0 开始）是哪个？

In [ ]:
ans_24_1 = int(mon[mon.flag].run.min()) if mon.flag.any() else None
check("练习 24.1", ans_24_1, int(mon[mon.flag].run.min()) if mon.flag.any() else None)

**练习 24.2**：把控制限从 3 倍改为 2 倍标准误（`z=2`)，会多出几个被标记的月份？这对误报意味着什么？

In [ ]:
mon2 = anchor_drift(hist, baseline_runs=4, z=2.0)
ans_24_2 = int(mon2.flag.sum() - mon.flag.sum())
check("练习 24.2", ans_24_2, int(mon2.flag.sum() - mon.flag.sum()))

## 2. 用哈希发现数据被改动

In [ ]:
from course.data import load_prompts
p = load_prompts()
h1 = sha256_of(p.to_dict(orient="list"))
p2 = p.copy(); p2.loc[0, "prompt"] = p2.loc[0, "prompt"] + " "      # 只多了一个空格
h2 = sha256_of(p2.to_dict(orient="list"))
print(h1[:16], h2[:16], "相同" if h1 == h2 else "不同")

## 3. 写一份可执行的规范

**练习 24.3**：把你的复现规范写成一个 Python 列表 `ans_24_3`，至少 6 条，每条是一句可以检查的要求（例如“每次评测保存 manifest.json，包含数据哈希”）。

In [ ]:
ans_24_3 = ["每次评测保存 manifest.json,含数据哈希", "记录被测模型名称、版本与采样参数", "记录裁判名称、版本、提示词全文与温度", "记录 git 提交号与依赖版本", "评测前先跑锚定集,超出控制限则暂停", "所有结果表注明数据来源与生成脚本"]
check_true("练习 24.3 至少 6 条可检查的要求", None if ans_24_3 is None else (isinstance(ans_24_3, list) and len(ans_24_3) >= 6))

## 口述题（录音，每题 90 秒）

1. 托管裁判在无通知的情况下更新，你怎么发现？
   <details><summary>参考要点</summary>每次评测前跑固定锚定集，用历史数据设控制限，超限即告警；记录裁判版本和日期；必要时用本地固定版本的模型做对照裁判。</details>

## 研究延伸

心理健康场景中，托管模型更新可能改变危机处理行为。用一个固定的锚定集（例如 McBain 等人那样的分级风险问题）长期监测多个模型，是一项有公共价值的研究。

In [ ]:
summary()